# Q3: Supplier Selection Prediction
## Can a model learn which suppliers we actually choose — and why?

**Report Section:** Question 3 · Classification  
**Data Source:** Supply Chain ML Predictive Modelling Report

## Objective

- Train a classifier to predict `selected_supplier_flag` (1/0) and learn which suppliers are actually chosen
- Audit whether selection logic aligns with stated procurement policy (detect divergence between policy and practice)
- Rank suppliers by selection probability to create an actionable shortlist for procurement teams

In [ ]:
# Step 0: Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
)

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load dataset
# df = pd.read_csv('supply_chain_data.csv')  # Replace with actual data path
# print(df.head())
# print(df.info())
# print(df['selected_supplier_flag'].value_counts())

## Implementation Plan

### Step 1: Examine Class Balance
- Count selected vs. non-selected using value_counts()
- Calculate class ratio; if >3:1, implement class_weight='balanced' in classifier
- Plot bar chart with percentage annotation

### Step 2: Feature Selection — What Drives Selection Decisions?
- Include all performance features: quality_score, supplier_reliability_score, defect_rate, return_rate, on_time_delivery_rate, delivery_time_days, lead_time_variance, price_per_unit, payment_term_days, offer_validity_days, delivery_mode, delivery_term_code, procurement_action_code
- Encode categoricals, scale numerics in ColumnTransformer pipeline
- Plot correlation bar chart with selected_supplier_flag

### Step 3: Logistic Regression Baseline
- Train LogisticRegression(max_iter=1000, class_weight='balanced') as interpretable baseline
- Evaluate with classification_report (precision, recall, F1 for both classes)
- Plot confusion matrix heatmap

### Step 4: Upgrade to Random Forest Classifier
- Train RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
- Compare F1-score (macro), ROC-AUC, precision-recall AUC against baseline
- Plot ROC curves for both models on same axes

### Step 5: Feature Importance — What Does the Model Think Drives Selection?
- Extract feature_importances_ from Random Forest
- Plot horizontal bar chart
- Cross-examine against domain knowledge: does model align with stated procurement policy?

### Step 6: Probability Calibration — Rank Suppliers by Selection Likelihood
- Use predict_proba() to generate selection probability (0–1) for each supplier
- Sort by probability descending
- Plot sorted scatter — 'Supplier Ranking by ML Selection Score'

### Step 7: Procurement Action Audit — Is Selection Logic Consistent?
- Group predicted probabilities by procurement_action_code and delivery_term_code
- Plot box plots of predicted probability per group
- Build summary table: procurement action → mean predicted selection probability

In [ ]:
# STEP 1: Examine Class Balance

# class_dist = df['selected_supplier_flag'].value_counts()
# class_pct = df['selected_supplier_flag'].value_counts(normalize=True) * 100

# print("Class Distribution:")
# print(class_dist)
# print(f"\nPercentages:")
# for idx, val in class_pct.items():
#     label = "Selected" if idx == 1 else "Not Selected"
#     print(f"  {label}: {val:.1f}%")

# # Check if class_weight='balanced' is needed
# ratio = max(class_dist) / min(class_dist)
# print(f"\nClass imbalance ratio: {ratio:.2f}:1")
# if ratio > 3:
#     print("⚠️ Significant imbalance detected. Using class_weight='balanced' in classifier.")

In [ ]:
# STEP 1 Visualization: Class Distribution Bar Chart

# fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# labels = ['Not Selected', 'Selected']
# colors = ['#ff7f0e', '#2ca02c']
# bars = ax.bar(labels, class_dist.values, color=colors, alpha=0.7, edgecolor='black')

# # Add percentage annotations
# for i, (bar, pct) in enumerate(zip(bars, class_pct.values)):
#     height = bar.get_height()
#     ax.text(bar.get_x() + bar.get_width()/2., height,
#             f'{pct:.1f}%\n(n={int(height)})',
#             ha='center', va='bottom', fontsize=11, fontweight='bold')

# ax.set_ylabel('Count')
# ax.set_title('Class Distribution: Supplier Selection')
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 2: Feature Selection — What Drives Selection Decisions?

# numerical_features = [
#     'quality_score', 'supplier_reliability_score', 'defect_rate', 'return_rate',
#     'on_time_delivery_rate', 'delivery_time_days', 'lead_time_variance',
#     'price_per_unit', 'payment_term_days', 'offer_validity_days'
# ]
# categorical_features = ['delivery_mode', 'delivery_term_code', 'procurement_action_code']

# all_features = numerical_features + categorical_features

# X = df[all_features]
# y = df['selected_supplier_flag']

# # Correlation analysis
# correlation_with_selection = df[numerical_features + ['selected_supplier_flag']].corr()['selected_supplier_flag']
# correlation_with_selection = correlation_with_selection.drop('selected_supplier_flag').sort_values()

# print("\nCorrelation with Supplier Selection:")
# print(correlation_with_selection)

In [ ]:
# STEP 2 Visualization: Correlation Bar Chart

# fig, ax = plt.subplots(figsize=(10, 6))
# colors = ['red' if x < 0 else 'green' for x in correlation_with_selection.values]
# ax.barh(correlation_with_selection.index, correlation_with_selection.values, color=colors, alpha=0.7)
# ax.set_xlabel('Correlation with Selected Supplier Flag')
# ax.set_title('Feature Correlation with Supplier Selection')
# ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
# ax.grid(axis='x', alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 3: Logistic Regression Baseline

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Create preprocessing pipeline
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', StandardScaler(), numerical_features),
#         ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
#     ])

# # Logistic Regression Baseline
# lr_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
# ])

# lr_pipeline.fit(X_train, y_train)
# y_pred_lr = lr_pipeline.predict(X_test)
# y_pred_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]

# # Classification report
# print("Logistic Regression - Classification Report:")
# print(classification_report(y_test, y_pred_lr, target_names=['Not Selected', 'Selected']))

# # AUC score
# roc_auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
# print(f"\nROC-AUC Score (LR): {roc_auc_lr:.3f}")

In [ ]:
# STEP 3 Visualization: Confusion Matrix (Logistic Regression)

# cm_lr = confusion_matrix(y_test, y_pred_lr)

# fig, ax = plt.subplots(figsize=(8, 6))
# sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'Count'})
# ax.set_xlabel('Predicted')
# ax.set_ylabel('Actual')
# ax.set_title('Confusion Matrix - Logistic Regression')
# ax.set_xticklabels(['Not Selected', 'Selected'])
# ax.set_yticklabels(['Not Selected', 'Selected'])
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 4: Upgrade to Random Forest Classifier

# rf_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1))
# ])

# rf_pipeline.fit(X_train, y_train)
# y_pred_rf = rf_pipeline.predict(X_test)
# y_pred_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# # Classification report
# print("Random Forest - Classification Report:")
# print(classification_report(y_test, y_pred_rf, target_names=['Not Selected', 'Selected']))

# # AUC score
# roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
# print(f"\nROC-AUC Score (RF): {roc_auc_rf:.3f}")

In [ ]:
# STEP 4 Visualization: ROC Curves (Logistic Regression vs. Random Forest)

# fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
# fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)

# fig, ax = plt.subplots(figsize=(10, 8))
# ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_lr:.3f})', linewidth=2)
# ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_rf:.3f})', linewidth=2)
# ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
# ax.set_xlabel('False Positive Rate')
# ax.set_ylabel('True Positive Rate')
# ax.set_title('ROC Curves: Model Comparison')
# ax.legend(loc='lower right', fontsize=11)
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 5: Feature Importance — What Does the Model Think Drives Selection?

# rf_model = rf_pipeline.named_steps['model']

# # Feature names (simplified; adjust based on actual post-encoding names)
# feature_names = all_features

# importance_df = pd.DataFrame({
#     'Feature': feature_names[:len(rf_model.feature_importances_)],
#     'Importance': rf_model.feature_importances_
# }).sort_values('Importance', ascending=True).tail(10)

# print("\nTop 10 Features Driving Supplier Selection (Random Forest):")
# print(importance_df)

# # Check alignment with stated procurement policy
# top_features = importance_df.tail(5)['Feature'].tolist()
# print(f"\n⚠️ Policy Audit: Top 5 model features are {top_features}")
# print("   Verify these align with your stated procurement policy.")

In [ ]:
# STEP 5 Visualization: Feature Importance

# fig, ax = plt.subplots(figsize=(10, 6))
# ax.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
# ax.set_xlabel('Importance Score')
# ax.set_title('Top 10 Features Driving Supplier Selection')
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 6: Probability Calibration — Rank Suppliers by Selection Likelihood

# # Generate selection probabilities for all test samples
# selection_probs = y_pred_proba_rf

# # Create ranking dataframe
# ranking_df = pd.DataFrame({
#     'Index': np.arange(len(selection_probs)),
#     'Selection_Probability': selection_probs,
#     'Actual_Selection': y_test.values,
#     'Predicted_Selection': y_pred_rf
# }).sort_values('Selection_Probability', ascending=False).reset_index(drop=True)

# print("\nTop 10 Suppliers by Predicted Selection Probability:")
# print(ranking_df.head(10))
# print("\nBottom 10 Suppliers by Predicted Selection Probability:")
# print(ranking_df.tail(10))

In [ ]:
# STEP 6 Visualization: Supplier Ranking

# fig, ax = plt.subplots(figsize=(12, 6))
# ax.scatter(ranking_df.index, ranking_df['Selection_Probability'], 
#            c=ranking_df['Actual_Selection'], cmap='RdYlGn', s=50, alpha=0.7, edgecolors='black')
# ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')
# ax.set_xlabel('Supplier Rank')
# ax.set_ylabel('Selection Probability')
# ax.set_title('Supplier Ranking by ML Selection Score')
# ax.legend()
# ax.grid(True, alpha=0.3)

# # Add colorbar
# cbar = plt.colorbar(ax.collections[0], ax=ax)
# cbar.set_label('Actual Selection')
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 7: Procurement Action Audit — Is Selection Logic Consistent?

# # Create full results dataframe with probability scores
# X_test_reset = X_test.reset_index(drop=True)
# results_df = X_test_reset.copy()
# results_df['predicted_selection_prob'] = y_pred_proba_rf
# results_df['actual_selection'] = y_test.values

# # Group by procurement action
# action_analysis = results_df.groupby('procurement_action_code')['predicted_selection_prob'].agg(['mean', 'std', 'count'])
# action_analysis = action_analysis.sort_values('mean', ascending=False)

# print("\nMean Selection Probability by Procurement Action:")
# print(action_analysis)

# # Group by delivery term
# term_analysis = results_df.groupby('delivery_term_code')['predicted_selection_prob'].agg(['mean', 'std', 'count'])
# term_analysis = term_analysis.sort_values('mean', ascending=False)

# print("\nMean Selection Probability by Delivery Term:")
# print(term_analysis)

In [ ]:
# STEP 7 Visualization: Procurement Action Box Plots

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# # By procurement action
# results_df.boxplot(column='predicted_selection_prob', by='procurement_action_code', ax=axes[0])
# axes[0].set_xlabel('Procurement Action Code')
# axes[0].set_ylabel('Predicted Selection Probability')
# axes[0].set_title('Selection Probability by Procurement Action')
# plt.sca(axes[0])
# plt.xticks(rotation=45)

# # By delivery term
# results_df.boxplot(column='predicted_selection_prob', by='delivery_term_code', ax=axes[1])
# axes[1].set_xlabel('Delivery Term Code')
# axes[1].set_ylabel('Predicted Selection Probability')
# axes[1].set_title('Selection Probability by Delivery Term')
# plt.sca(axes[1])
# plt.xticks(rotation=45)

# fig.suptitle('Selection Logic Consistency Audit', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.show()

## Key Takeaways

Once all steps are complete, your notebook will deliver:

1. **Class Balance Assessment** — Detection of imbalance and application of class_weight='balanced' if needed
2. **Correlation Baseline** — Which single features most strongly correlate with supplier selection
3. **Model Comparison** — Logistic regression baseline vs. random forest, with ROC-AUC as the discriminator
4. **Feature Importance Audit** — Top 5 features driving selection; compare against stated procurement policy to detect misalignment
5. **Supplier Ranking** — Continuous probability scores that transform binary model into actionable supplier shortlist
6. **Consistency Check** — Mean selection probability by procurement action and delivery term; identifies underperforming strategies

The most strategically valuable finding is **misalignment between model importance and stated policy** — it proves selection decisions are not what management thinks they are.